把原始数据提取需要的

In [3]:
import pandas as pd

# 输入输出文件路径
input_file = f"C:/Users/User/Desktop/pythondata/预后模型评估/原始数据/增加的新的数据.xlsx"
output_file = f"C:/Users/User/Desktop/pythondata/预后模型评估/zs/output1增加的新的数据.xlsx"

# 读取Excel文件
df = pd.read_excel(input_file)

# 初始化一个空列表用于存储结果
result = []

# 定义处理每个 PET 组的函数
def process_pet_group(row, pet_prefix):
    treatment_col = f'{pet_prefix}检查号'
    if pd.notna(row[treatment_col]):
        new_row = {
            '姓名': row['姓名'],
            '病历号': row['病历号'],#病历号
            '检查号': row[treatment_col],
            '治疗前后/复发': row[f'{pet_prefix}治疗前后/复发（1/2/3）'],
            '检查时间': row[f'{pet_prefix}检查时间'],
            '影像表现': row[f'{pet_prefix}影像表现'],
            '诊断结论': row[f'{pet_prefix}诊断结论']
        }
        result.append(new_row)

# 遍历每一行
for _, row in df.iterrows():
    process_pet_group(row, 'PET1')
    process_pet_group(row, 'PET2')
    process_pet_group(row, 'PET3')
    process_pet_group(row, 'PET4')
    process_pet_group(row, 'PET5')
    process_pet_group(row, 'PET6')

# 转换为DataFrame并写入新文件
result_df = pd.DataFrame(result)
result_df.to_excel(output_file, index=False)

print(f"已成功提取数据到文件：{output_file}")

已成功提取数据到文件：C:/Users/User/Desktop/pythondata/预后模型评估/zs/output1增加的新的数据.xlsx


In [ ]:
#deepseek qwenplus
import pandas as pd
import json
import re

# 配置参数 内部外部
#excel_file = r"C:\Users\User\Desktop\pythondata\预后模型评估\10112\bidui\jiegouh\多模型评测\结构化表格\匹配及关联数据.xlsx"         # Excel 文件路径
#excel_file = r"C:\Users\User\Desktop\pythondata\预后模型评估\10112\bidui\jiegouh\多模型评测\结构化表格\匹配及关联数据.xlsx"         # Excel 文件路径

sheet_name = "Sheet1"             # 工作表名称
      # 输出文件路径
col1 = "影像表现"                 # 第一列列名
col2 = "诊断结论"                   # 第二列列名


# 读取 Excel 数据
df = pd.read_excel(excel_file, sheet_name=sheet_name)
# with open("xmoutput.md", "r", encoding="utf-8") as file:
#     markdown_content = file.read()  

markdown_content =  """
下面是PET-CT报告的专业术语定义用来学习参考。

扫描区包括四肢，脊柱，颅骨。
在后面的填写规范中NA也表示不确定、不明，X4和NA都可指示多发

阳性信息  填写规范：阳性/阴性
PET阳性（PET-positive）的定义
满足以下任一条，即判定为阳性病灶：
1.局灶性骨病变：
  a.在基线或随访扫描中，出现 局灶性 FDG 摄取，且其 SUVmax ≥ 纵隔血池（mediastinal blood pool SUV）或 ≥ 周围正常骨髓组织；
  b.可伴或不伴 CT 可见的溶骨性破坏。
2.髓外病变（EM）或髓旁病灶(PM)：
  a.任何软组织或骨旁区域出现 新发或持续的高代谢灶（SUVmax 标准同上）。
3.弥漫性骨髓高代谢：
  a.骨髓弥漫性 FDG 摄取高于纵隔血池，且无其他可解释原因（如感染、骨折）
PET阴性（PET-negative）的定义
必须 同时满足 以下所有条件：
1.完全消失：
  a.与基线相比，所有先前的高代谢病灶（包括局灶性骨病变、EMD、弥漫性骨髓摄取） 完全消失；
2.或代谢降至背景以下：
  a.残余灶的SUVmax < 纵隔血池 SUV 或 < 周围正常骨髓组织；
3.无新发病灶：
  a.无新发局灶性或弥漫性高代谢灶；
4.无新骨质破坏：
  a.CT部分未见新发溶骨性病变。
对于初诊患者,PET-CT阴性指的是不满足PET-CT阳性任一条件。

骨髓/骨骼整体代谢活性 填写规范：是/否 数值 是/否
骨髓/骨骼整体代谢活性增高，主要看SUV值>肝脏代谢，通常以肝脏代谢活性为参照标准。需排除化疗后骨髓增生反应或生长因子使用导致的骨髓高代谢（可能表现为弥漫性摄取增高）。

骨质破坏 填写规范：有/无
骨质破坏指的是PET-CT中显示明确的溶骨性病变（如骨皮质缺损、溶骨性破坏、穿凿样改变）。


按 MM 专用 Deauville 5 级标准： 
1 分：病灶处完全无摄取，或肉眼不可辨。
2 分：可见摄取，但病灶 SUVmax ≤ 肝脏 SUVmax。
3 分：病灶 SUVmax ＞ 肝脏，但增幅 ≤ 10 %。
4 分：病灶 SUVmax ＞ 肝脏 +10 %，但未达 2 倍。
5 分：病灶 SUVmax ≥ 2× 肝脏，或出现任何新的高摄取灶。
放射性摄取增高指的是比周围骨髓组织高，或高于肝脏基础摄取值。

病变类别有局灶性病灶（FLs），髓外病灶(EM)，髓旁病灶(PM)，溶骨性病变(L)。
局灶性病灶（FLs）填写规范：S/SP/Ex-Sp X1/X2/X3/X4 X1/X2/X3/X4 数值 数值 
局灶性病变部位分为S (Skull) SP (spine) Ex-Sp (extra-spine)，局灶性病变数量分为X1 (None) 、X2 (N =1 to 3)、 X3 (N =4 to 10)、 X4 (N >10),填写时应填写X1或X2或X3或X4。局灶性病变定义为在至少 2 个连续的 PET 切片上或大小大于5mm可见的 F-FDG 摄取增加的局灶区域，摄取增加指的是SUVmax值大于周围骨髓/骨骼组织或高于肝脏组织或大于2.5，需表现为局灶性（非弥漫性），即单个或多个离散的高摄取灶。需排除弥漫性摄取和生理学摄取（炎性摄取）。溶骨性病变是指通过CT成像检测到的骨质破坏，表现为骨皮质或骨小梁的缺失，通常提示骨髓瘤细胞浸润导致的骨吸收和骨结构破坏。
髓外病灶(EM) 填写规范： 文本 N或EN或N/EN X1/X2/X3/X4 X1/X2/X3/X4 数值
髓外病变（EM）指的是软组织/器官高摄取灶，髓外病变（EM）分为N(淋巴结)/EN(非淋巴结)。髓外病变淋巴结如果为高摄取灶，但诊断结论为随访、排除炎症、其他肿瘤性病变等则考虑髓外病变；对于非淋巴结组织，若诊断结论仅为随访考虑不是髓外病变。髓外病变（EM）部位填写时应填写N或EN或N/EN，当部位填写N/EN时数量可填写如X4/X2形式。
髓旁病灶(PM) 填写规范： 文本 X1/X2/X3/X4 X1/X2/X3/X4 数值 数值
髓旁病灶(PM)是指软组织肿块从骨髓生长到周围组织。髓外病变和髓旁病灶需要排除退行性病变、炎性病变以及非多发性骨髓瘤的肿瘤病变（影像表现和诊断结论作为依据）。
特别注意旁髓病变，临近骨且形成软组织肿块。

溶骨性病变(L) :溶骨性病变是指通过CT成像检测到的骨质破坏，表现为骨皮质或骨小梁的缺失，通常提示骨髓瘤细胞浸润导致的骨吸收和骨结构破坏。

SUVmax最大值 填写规范：数值 
SUVmax最大值指的是骨髓瘤病变，包括弥漫性病变，局灶性病变，髓外病变，髓旁病变。

骨折 填写规范：有/无 S/SP/Ex-Sp 新发/陈旧 是/否
骨折指CT病理性骨折，骨折部位分为S (Skull) SP (spine) Ex-Sp (extra-spine)

骨骼系统外科手术证据 填写规范：有/无 文本 a/b/c/d
骨骼系统外科手术证据影像学报告中需明确标注 既往或近期骨骼系统手术痕迹，其类型分为：
  a.内固定植入物（如钢板、螺钉、髓内钉）
  b.椎体成形术/后凸成形术（骨水泥填充）
  c.骨移植或人工关节置换
  d.其他手术相关结构改变（如截骨术、刮除术痕迹）

所有数据如果是无法判断或无法获取填写规范即写NA

最终输出结构化表格17行6列（待填写）如下：
| 阳性信息                     | PET阳性/阴性   |            |                       |                    |              |
|-----------------------------|----------------|------------|-----------------------|--------------------|--------------|
|                             |  （待填写）      |            |                       |                    |              |
| 骨髓/骨骼整体代谢活性         | 是否增高       | SUVmax值   | 长骨的高代谢（是/否）   |                    |              |
|                             |    （待填写）    | （待填写）   | （待填写）           |                    |              |
| 骨质破坏                     | 有/无          |            |                       |                    |              |
|                             |  （待填写）      |            |                       |                    |              |
| 病变类别                     | 部位           | 数量         | 溶骨性病变数量         | Deauville评分      | SUVmax值     |
| 局灶性病灶（FLs）            |  （待填写）     |   （待填写）    |（待填写）           | （待填写）         |  （待填写）     |
| 髓外病灶(EM)                 |  （待填写）      |  （待填写）   |  （待填写）           |  （待填写）         |  （待填写）      |
| 髓旁病灶(PM)                |   （待填写）     |  （待填写）   | （待填写）            |  （待填写）           |  （待填写）     |
|                             | SUVmax最大值   |            |                      |                      |            |
| SUVmax最大值                |  （待填写）   |             |                       |                      |            |
|                            | 有/无           | 部位        | 新发/陈旧              | 多发性骨髓瘤是否引起  |              |   
| 骨折                        | （待填写）     |  （待填写）   |   （待填写）        |  （待填写）        |             |
|                            | 有/无           | 部位        | 类型                  |                     |              |   
| 骨骼系统外科手术证据        | （待填写）       |   （待填写）    |  （待填写）           |                      |             |

真实示例3例：
a
| 阳性信息                     | PET阳性/阴性   |            |                       |                    |              |
|-----------------------------|----------------|------------|-----------------------|--------------------|--------------|
|                             | 阳性         |            |                       |                    |              |
| 骨髓/骨骼整体代谢活性         | 是否增高       | SUVmax值   | 长骨的高代谢（是/否）   |                    |              |
|                             | 是             | 2.7        | 是                    |                    |              |
| 骨质破坏                     | 有/无          |            |                       |                    |              |
|                             | 有             |            |                       |                    |              |
| 病变类别                     | 部位           | 数量         | 溶骨性病变数量         | Deauville评分      | SUVmax值     |
| 局灶性病灶（FLs）            | S/SP/Ex-Sp | X4        | NA                    | NA                 | 9.8          |
| 髓外病灶(EM)                 | N/EN           | X4/X2         | NA                    | NA                 | 3.6          |
| 髓旁病灶(PM)                | 右侧股骨       | 1          | NA                    | NA                 | 6.6          |
|                             | SUVmax最大值   |            |                      |                      |            |
| SUVmax最大值                | 9.8          |             |                       |                      |            |
|                            | 有/无           | 部位        | 新发/陈旧              | 多发性骨髓瘤是否引起  |              |   
| 骨折                        | 无             | NA         | NA                    | NA                 |             |
|                            | 有/无           | 部位        | 类型                  |                     |              |   
| 骨骼系统外科手术证据        | 无             | NA         | NA                    |                      |             |

b
| 阳性信息                     | PET阳性/阴性   |            |                       |                    |              |
|-----------------------------|----------------|------------|-----------------------|--------------------|--------------|
|                             | 阳性         |            |                       |                    |              |
| 骨髓/骨骼整体代谢活性         | 是否增高       | SUVmax值   | 长骨的高代谢（是/否）   |                    |              |
|                             | 是             | 2.7        | 是                    |                    |              |
| 骨质破坏                     | 有/无          |            |                       |                    |              |
|                             | 有             |            |                       |                    |              |
| 病变类别                     | 部位           | 数量         | 溶骨性病变数量         | Deauville评分      | SUVmax值     |
| 局灶性病灶（FLs）            | S/SP/Ex-Sp | X4        | NA                    | NA                 | 9.8          |
| 髓外病灶(EM)                 | N/EN           | X4/X2         | NA                    | NA                 | 3.6          |
| 髓旁病灶(PM)                | 右侧股骨       | 1          | NA                    | NA                 | 6.6          |
|                             | SUVmax最大值   |            |                      |                      |            |
| SUVmax最大值                | 9.8          |             |                       |                      |            |
|                            | 有/无           | 部位        | 新发/陈旧              | 多发性骨髓瘤是否引起  |              |   
| 骨折                        | 无             | NA         | NA                    | NA                 |             |
|                            | 有/无           | 部位        | 类型                  |                     |              |   
| 骨骼系统外科手术证据        | 无             | NA         | NA                    |                      |             |

c
| 阳性信息                     | PET阳性/阴性   |            |                       |                    |              |
| ---------------------------- | -------------- | ---------- | --------------------- | ------------------ | ------------ |
|                             | 阳性         |            |                       |                    |              |
| 骨髓/骨骼整体代谢活性         | 是否增高       | SUVmax值   | 长骨的高代谢（是/否）   |                    |              |
|                             | 是             | 2.7        | 是                    |                    |              |
| 骨质破坏                     | 有/无          |            |                       |                    |              |
|                             | 无             |            |                       |                    |              |
| 病变类别                     | 部位           | 数量         | 溶骨性病变数量         | Deauville评分      | SUVmax值     |
| 局灶性病灶（FLs）            | NA             | X1         | 0                     | NA                 | NA           |
| 髓外病灶(EM)                 | NA             | NA         | NA                    | NA                 | NA           |
| 髓旁病灶(PM)                | NA             | NA         | NA                    | NA                 | NA           |
|                             | SUVmax最大值   |            |                       |                    |              |
| SUVmax最大值                | 2.7            |            |                       |                    |              |
|                            | 有/无           | 部位        | 新发/陈旧              | 多发性骨髓瘤是否引起  |              |   
| 骨折                        | 无             | NA         | NA                    | NA                 |              |
|                            | 有/无           | 部位        | 类型                  |                     |              |   
| 骨骼系统外科手术证据        | 无             | NA         | NA                    |                     |              |
"""
# 生成 JSON 数据
# def process_row(row):
#     return f"{delimiter1}{row[col1]}{delimiter2}{row[col2]}" + "请生成对应的且格式一致的示例输出表格。"
mark =  "你是放射科专家，擅长进行PET-CT报告解读。"
json_obj = []
for idx, row in df.iterrows():
        # 拼接两列内容
    user_content = f"PET-CT报告如下\n" + f"影像表现：{row[col1]}  诊断结论：{row[col2]}" + "多发性骨髓瘤PET-CT自由报告如上所述" + "/n我的请求是将结构化表格（待填写）部分填写完整，并严格按照其格式输出，仅输出17行6列的结构化表格。"
        # 构建 JSON 对象
    json_obj.append({
        "custom_id": str(idx + 1),
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": "qwen3",
            "messages":[
                {"role": "user", "content": mark + markdown_content + user_content}
            ],"temperature": 0.7, "top_p": 0.9}
    })
        
        # 写入文件（JSON Lines 格式）
    with open("loraqwen3结构化表格输入prompt.jsonl", "w", encoding="utf-8") as f:    
        for item in json_obj:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"生成完成，文件已保存至 ")

生成完成，文件已保存至 


jsonl排序

In [4]:
import json

def sort_jsonl_by_custom_id(input_file, output_file):
    """
    根据 custom_id 的数值顺序重新排列 JSONL 文件
    """
    # 读取所有行
    lines = []
    with open(input_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:  # 跳过空行
                try:
                    data = json.loads(line)
                    custom_id = data.get("custom_id", "")
                    
                    # 检查 custom_id 是否为纯数值
                    if custom_id.isdigit():
                        numeric_id = int(custom_id)
                    else:
                        try:
                            numeric_id = float(custom_id)
                        except ValueError:
                            # 如果不是纯数值，跳过或设置一个默认值
                            numeric_id = float('inf')  # 或者可以跳过这个记录
                            
                    lines.append((numeric_id, line))
                except json.JSONDecodeError:
                    print(f"跳过无效JSON行: {line}")
                    continue
    
    # 按照 numeric_id 排序
    lines.sort(key=lambda x: x[0])
    
    # 写入排序后的结果
    with open(output_file, 'w', encoding='utf-8') as f:
        for _, line in lines:
            f.write(line + '\n')

# 使用示例
sort_jsonl_by_custom_id('qwen3.jsonl', 'qwen3_sorted.jsonl')

In [5]:
import json
import re
import pandas as pd

# 文件路径
jsonl_file = "outputjghjg_sorted.jsonl"
excel_file = "jghjg.xlsx"
sheet_name = "Sheet1"
target_column = "md-表格"

# 读取 JSONL 文件并提取 content
contents = []
with open(jsonl_file, "r", encoding="utf-8") as f:
    for line in f:
        try:
            data = json.loads(line.strip())
            # 提取 content 内容
            content = data.get("response", {}).get("body", {}).get("choices", [{}])[0].get("message", {}).get("content", "")
            # 截断 |\n\n 之后的内容
            processed_content = re.split(r'\n\n', content)[0]
            contents.append(content)
        except Exception as e:
            print(f"解析 JSON 行出错：{e}")
            contents.append("")

# 读取 Excel 文件
try:
    df = pd.read_excel(excel_file, sheet_name=sheet_name)
except FileNotFoundError:
    # 若文件不存在，创建一个空 DataFrame
    df = pd.DataFrame()

# 如果 deep0seek-md 列不存在，先创建该列
if target_column not in df.columns:
    df[target_column] = None

# 确保内容长度不超过现有行数，否则扩展行
max_len = max(len(contents), len(df))
df = df.reindex(range(max_len))

# 将处理后的内容按顺序写入 deepseek-md 列
for i, content in enumerate(contents):
    if i < len(df):
        df.at[i, target_column] = content
    else:
        df.loc[i, target_column] = content

# 保存回 Excel 文件
df.to_excel(excel_file, index=False, sheet_name=sheet_name, engine='openpyxl')

print(f"处理完成，结果已追加至 {excel_file} 的 '{target_column}' 列")

处理完成，结果已追加至 jghjg.xlsx 的 'md-表格' 列


In [3]:
import pandas as pd

def clean_invalid_cells_simple(file_path, target_column, sheet_name=0):
    """
    简化版本：清理指定列中不包含有效内容的单元格
    """
    # 有效内容列表
    VALID_CONTENTS = [
        "(FLs)影像表现", "(EM)影像表现", "(PM)影像表现", 
        "骨折影像表现", "骨折诊断结论", "(L)影像表现", 
        "(L)诊断结论", "(FLs)诊断结论", "(EM)诊断结论", "(PM)诊断结论"
    ]
    
    try:
        # 读取Excel文件
        df = pd.read_excel(file_path, sheet_name=sheet_name)
        
        print(f"原文件形状: {df.shape}")
        print(f"'{target_column}' 列非空值数量: {df[target_column].notna().sum()}")
        
        # 清理无效单元格
        def should_clear_cell(value):
            if pd.isna(value) or value == "":
                return False
            cell_str = str(value).strip()
            return not any(valid in cell_str for valid in VALID_CONTENTS)
        
        # 应用清理
        mask = df[target_column].apply(should_clear_cell)
        cleared_count = mask.sum()
        
        df.loc[mask, target_column] = ""
        
        # 保存文件
        output_path = file_path.replace('.xlsx', '_cleaned.xlsx')
        df.to_excel(output_path, index=False)
        
        print(f"清理完成! 共清空 {cleared_count} 个无效单元格")
        print(f"新文件已保存: {output_path}")
        
        # 显示被清理的内容示例
        if cleared_count > 0:
            print("\n被清理的内容示例:")
            original_df = pd.read_excel(file_path, sheet_name=sheet_name)
            cleared_samples = original_df.loc[mask, target_column].head(5)
            for sample in cleared_samples:
                print(f"  - '{sample}'")
        
        return True
        
    except Exception as e:
        print(f"错误: {e}")
        return False

# 使用示例
if __name__ == "__main__":
    # 替换为你的实际文件路径和列名
    clean_invalid_cells_simple("C:/Users/User/Desktop/pythondata/预后模型评估/1013/病灶明细表与结构化表格初诊基线MRD金标准.xlsx", "病灶明细表")

原文件形状: (364, 14)
'病灶明细表' 列非空值数量: 363
清理完成! 共清空 37 个无效单元格
新文件已保存: C:/Users/User/Desktop/pythondata/预后模型评估/1013/病灶明细表与结构化表格初诊基线MRD金标准_cleaned.xlsx

被清理的内容示例:
  - '|'
  - '|编号|部位|SUVmax|CT表现|
|---|---|---|---|
'
  - '|编号|部位|SUVmax|CT表现|
|---|---|---|---|
'
  - '|编号|部位|SUVmax|CT表现|
|---|---|---|---|
'
  - '|编号|部位|SUVmax|CT表现|
|---|---|---|---|
'


In [ ]:
import pandas as pd
import re

def fix_merged_md_lines(table_text):
    """
    专门处理MD表格中合并的行
    """
    valid_codes = [
        "(FLs)影像表现", "(EM)影像表现", "(PM)影像表现", 
        "骨折影像表现", "骨折诊断结论", "(L)影像表现", 
        "(L)诊断结论", "(FLs)诊断结论", "(EM)诊断结论", "(PM)诊断结论"
    ]
    
    lines = table_text.strip().split('\n')
    
    # 必须有表头和分隔行
    if len(lines) < 2:
        print("错误：表格格式不完整")
        return None
    
    # 处理数据行
    fixed_lines = lines[:2]  # 保留表头和分隔行
    
    for i in range(2, len(lines)):
        line = lines[i].strip()
        if not line:
            continue
            
        # 检查是否有多个有效编号（表明有合并）
        found_codes = [code for code in valid_codes if code in line]
        
        if len(found_codes) <= 1:
            fixed_lines.append(line)
        else:
            print(f"发现合并行，正在分割: {line}")
            
            # 简单分割：在每个有效编号前分割（除了第一个）
            parts = []
            current_line = line
            
            for code in found_codes[1:]:  # 跳过第一个编号
                if code in current_line:
                    # 找到第二个及以后的编号位置
                    pos = current_line.find(code)
                    if pos > 0:
                        # 分割为两部分
                        part1 = current_line[:pos].strip()
                        part2 = current_line[pos:].strip()
                        if part1:
                            parts.append(part1)
                        current_line = part2
            
            # 添加最后一部分
            if current_line:
                parts.append(current_line)
            
            fixed_lines.extend(parts)
            print(f"分割为 {len(parts)} 行")
    
    return '\n'.join(fixed_lines)

def process_md_table(input_text):
    """
    处理MD表格的主函数
    """
    try:
        result = fix_merged_md_lines(input_text)
        if result:
            return result
        else:
            print("表格处理失败")
            return input_text
    except Exception as e:
        print(f"处理过程中出错: {e}")
        return input_text

def process_xlsx_md_tables(file_path, column_name, sheet_name=0):
    """
    处理xlsx文件中指定列的MD表格
    
    参数:
    file_path: xlsx文件路径
    column_name: 包含MD表格的列名
    sheet_name: 工作表名称或索引
    """
    try:
        # 读取Excel文件
        df = pd.read_excel(file_path, sheet_name=sheet_name)
        
        # 检查列是否存在
        if column_name not in df.columns:
            print(f"错误: 列 '{column_name}' 在文件中不存在")
            print(f"可用的列: {list(df.columns)}")
            return False
        
        print(f"开始处理文件: {file_path}")
        print(f"处理列: {column_name}")
        print(f"总行数: {len(df)}")
        
        processed_count = 0
        fixed_count = 0
        
        # 遍历指定列的每个单元格
        for index, value in df[column_name].items():
            if pd.isna(value) or value == "":
                continue
                
            cell_text = str(value).strip()
            
            # 检查是否是MD表格（包含表头）
            if "|编号|部位|SUVmax|CT表现|" in cell_text:
                processed_count += 1
                print(f"\n处理第 {index + 1} 行:")
                print("原始内容:")
                print(cell_text)
                
                # 处理MD表格
                fixed_table = process_md_table(cell_text)
                
                if fixed_table != cell_text:
                    fixed_count += 1
                    # 更新单元格内容
                    df.at[index, column_name] = fixed_table
                    print("修复后的内容:")
                    print(fixed_table)
                else:
                    print("无需修复")
        
        # 保存处理后的文件
        output_path = file_path.replace('.xlsx', '_fixed.xlsx')
        df.to_excel(output_path, index=False)
        
        print(f"\n处理完成!")
        print(f"共处理 {processed_count} 个MD表格")
        print(f"修复了 {fixed_count} 个表格")
        print(f"新文件已保存为: {output_path}")
        
        return True
        
    except Exception as e:
        print(f"处理文件时出错: {e}")
        return False

# 使用示例
if __name__ == "__main__":
    # 方法1: 处理指定列中的所有MD表格
    file_path = "C:/Users/User/Desktop/pythondata/预后模型评估/1013/病灶明细表与结构化表格初诊基线MRD金标准_cleaned.xlsx"  # 替换为你的文件路径
    column_name = "病灶明细表"  # 替换为包含MD表格的列名
    
    process_xlsx_md_tables(file_path, column_name)
    
    # 方法2: 处理所有包含MD表格表头的单元格
    # process_xlsx_by_cell_content(file_path, "|编号|部位|SUVmax|CT表现|")
    
    # 演示示例
    print("使用方法:")
    print("1. 处理指定列: process_xlsx_md_tables('data.xlsx', '报告内容')")
    print("2. 搜索所有列: process_xlsx_by_cell_content('data.xlsx', '|编号|部位|SUVmax|CT表现|')")

开始处理文件: C:/Users/User/Desktop/pythondata/预后模型评估/1013/病灶明细表与结构化表格初诊基线MRD金标准_cleaned.xlsx
处理列: 病灶明细表
总行数: 364

处理第 2 行:
原始内容:
|编号|部位|SUVmax|CT表现|
|---|---|---|---|
|(FLs)影像表现|扫描区全身（胸骨、双侧肋骨、双侧肩胛骨、四肢骨近端、脊柱及骨盆诸骨）、T6胸椎、左侧第3肋骨|3.8|扫描区全身（胸骨、双侧肋骨、双侧肩胛骨、四肢骨近端、脊柱及骨盆诸骨）骨质密度不均匀轻度减低，局部似见斑点状低密度影，放射性摄取不均匀增高，SUV最大值约3.8；T6胸椎、左侧第3肋骨可见斑片状放射性摄取增高，SUV最大值约8.5；|
|(FLs)诊断结论|扫描区全身、T6胸椎、左侧第3肋骨||扫描区全身骨质密度不均匀轻度减低，局部似见斑点状低密度影，FDG代谢增高，其中T6胸椎、左侧第3肋骨斑片状FDG代谢增高灶，结合临床及实验室检查，考虑多发性骨髓瘤；|
|(L)影像表现|扫描区全身（胸骨、双侧肋骨、双侧肩胛骨、四肢骨近端、脊柱及骨盆诸骨）|3.8|扫描区全身（胸骨、双侧肋骨、双侧肩胛骨、四肢骨近端、脊柱及骨盆诸骨）骨质密度不均匀轻度减低，局部似见斑点状低密度影，放射性摄取不均匀增高，SUV最大值约3.8；|
|(L)诊断结论|扫描区全身、T6胸椎、左侧第3肋骨||扫描区全身骨质密度不均匀轻度减低，局部似见斑点状低密度影，FDG代谢增高，其中T6胸椎、左侧第3肋骨斑片状FDG代谢增高灶，结合临床及实验室检查，考虑多发性骨髓瘤；|
|(EM)影像表现|腹膜后主动脉旁、右侧盆壁、双侧腹股沟区|11.1|腹膜后主动脉旁、右侧盆壁及双侧腹股沟区多发淋巴结显示，大者直径约1.3cm，放射性摄取增高，SUV最大值约11.1；|
|(EM)诊断结论|腹膜后主动脉旁、右侧盆壁、双侧腹股沟区||腹膜后主动脉旁、右侧盆壁及双侧腹股沟区多发淋巴结显示，FDG代谢增高，建议密切随访观察。|
无需修复

处理第 3 行:
原始内容:
|编号|部位|SUVmax|CT表现|
|---|---|---|---|

|(L)影像表现|双侧肱骨近端、锁骨、胸骨、肋骨、肩胛骨、脊柱椎体、骨盆诸骨、双侧股骨上段|2.2|扫描区骨骼（双侧肱骨近端、锁骨、胸骨、肋骨

In [ ]:
import pandas as pd
import re

def quick_validate_md_tables(file_path, column_name):
    """
    快速验证MD表格格式
    """
    try:
        df = pd.read_excel(file_path)
        
        if column_name not in df.columns:
            print(f"列 '{column_name}' 不存在")
            return
        
        valid_codes = [
            "(FLs)影像表现", "(EM)影像表现", "(PM)影像表现", 
            "骨折影像表现", "骨折诊断结论", "(L)影像表现", 
            "(L)诊断结论", "(FLs)诊断结论", "(EM)诊断结论", "(PM)诊断结论"
        ]
        
        print(f"快速验证结果 - 文件: {file_path}, 列: {column_name}")
        print("=" * 50)
        
        for index, value in df[column_name].items():
            if pd.isna(value):
                continue
                
            text = str(value).strip()
            if "|编号|部位|SUVmax|CT表现|" in text:
                lines = text.split('\n')
                
                # 基本格式检查
                has_header = len(lines) > 0 and "|编号|部位|SUVmax|CT表现|" in lines[0]
                has_separator = len(lines) > 1 and re.match(r'^\|[- ]+\|', lines[1])
                has_data = len(lines) > 2
                
                # 检查合并行
                merged_lines = 0
                if has_data:
                    for line in lines[2:]:
                        code_count = sum(1 for code in valid_codes if code in line)
                        if code_count > 1:
                            merged_lines += 1
                
                status = "✅ 正常" if has_header and has_separator and has_data and merged_lines == 0 else "⚠️  需检查"
                
                print(f"行 {index+1}: {status} | 表头: {has_header} | 分隔行: {has_separator} | 数据行: {has_data} | 合并行: {merged_lines}")
    
    except Exception as e:
        print(f"快速验证出错: {e}")

# 使用示例
if __name__ == "__main__":
    # 替换为你的实际文件路径和列名
    file_path = "C:/Users/User/Desktop/pythondata/预后模型评估/1013/病灶明细表与结构化表格初诊基线MRD金标准_cleaned_fixed.xlsx"
    column_name = "病灶明细表"
    
    print("MD表格格式验证工具")
    print("=" * 50)
    
    print("\n" + "=" * 50)
    print("快速验证模式:")
    
    # 快速验证
    quick_validate_md_tables(file_path, column_name)

MD表格格式验证工具

快速验证模式:
快速验证结果 - 文件: C:/Users/User/Desktop/pythondata/预后模型评估/1013/病灶明细表与结构化表格初诊基线MRD金标准_cleaned_fixed.xlsx, 列: 病灶明细表
行 2: ✅ 正常 | 表头: True | 分隔行: <re.Match object; span=(0, 5), match='|---|'> | 数据行: True | 合并行: 0
行 3: ✅ 正常 | 表头: True | 分隔行: <re.Match object; span=(0, 5), match='|---|'> | 数据行: True | 合并行: 0
行 4: ✅ 正常 | 表头: True | 分隔行: <re.Match object; span=(0, 5), match='|---|'> | 数据行: True | 合并行: 0
行 5: ✅ 正常 | 表头: True | 分隔行: <re.Match object; span=(0, 5), match='|---|'> | 数据行: True | 合并行: 0
行 6: ✅ 正常 | 表头: True | 分隔行: <re.Match object; span=(0, 5), match='|---|'> | 数据行: True | 合并行: 0
行 7: ✅ 正常 | 表头: True | 分隔行: <re.Match object; span=(0, 5), match='|---|'> | 数据行: True | 合并行: 0
行 8: ✅ 正常 | 表头: True | 分隔行: <re.Match object; span=(0, 5), match='|---|'> | 数据行: True | 合并行: 0
行 9: ✅ 正常 | 表头: True | 分隔行: <re.Match object; span=(0, 5), match='|---|'> | 数据行: True | 合并行: 0
行 10: ✅ 正常 | 表头: True | 分隔行: <re.Match object; span=(0, 5), match='|---|'> | 数据行: True | 合并行: 0
行 11: ✅ 正常 | 表头